# ActionFlow Notebook

This notebook is the primary entry point for the project. It walks through the full KTH optical-flow pipeline on CPU: configuration, cache-aware frame extraction, dense Farneback optical flow, dataset construction, model training, evaluation, and inline plots.

The notebook is intentionally self-contained for the pipeline logic. Only the model definition, trainer, and metrics utilities stay in Python modules so the notebook stays readable.

In [ ]:
%matplotlib inline

import json
import random
import re
import subprocess
import sys
from dataclasses import dataclass
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

sys.path.insert(0, str(Path('src').resolve()))

from actionflow.models.resnet_flow import build_resnet18_flow
from actionflow.training.metrics import (
    classification_report,
    compute_accuracy,
    compute_confusion_matrix,
    plot_confusion_matrix,
    plot_training_curves,
)
from actionflow.training.trainer import Trainer

plt.style.use('seaborn-v0_8-whitegrid')
torch.set_num_threads(max(torch.get_num_threads() // 2, 1))

## 1. Configuration

Everything starts with a small dataclass. The default values are chosen for a CPU-friendly notebook run: a modest subset per class, two epochs, and cache-aware preprocessing so existing frames and optical flow are reused instead of recomputed.

If you want a larger run, increase the per-class limits or set them to `None`.

In [ ]:
@dataclass
class ActionFlowConfig:
    data_root: Path = Path('data/kth')
    raw_root: Path = Path('data/kth/raw')
    download_script: Path = Path('download_kth.sh')
    class_names: tuple[str, ...] = (
        'boxing',
        'handclapping',
        'handwaving',
        'jogging',
        'running',
        'walking',
    )
    resize: tuple[int, int] = (224, 224)
    clip_length: int = 10
    frame_stride: int = 2
    train_clips_per_class: int | None = 6
    test_clips_per_class: int | None = 4
    mode: str = 'flow'
    pretrained_backbone: bool = True
    batch_size: int = 8
    epochs: int = 2
    lr: float = 1e-3
    weight_decay: float = 1e-4
    scheduler: str = 'cosine'
    device: str = 'cpu'
    num_workers: int = 0
    seed: int = 42
    output_dir: Path = Path('outputs/notebook')

    def __post_init__(self) -> None:
        if self.mode not in {'flow', 'rgb'}:
            raise ValueError("mode must be 'flow' or 'rgb'.")
        self.num_classes = len(self.class_names)
        self.input_channels = self.clip_length * 2 if self.mode == 'flow' else 3


config = ActionFlowConfig()
config.output_dir.mkdir(parents=True, exist_ok=True)

display(
    pd.DataFrame(
        {
            'value': {
                'data_root': str(config.data_root),
                'raw_root': str(config.raw_root),
                'download_script': str(config.download_script),
                'class_names': config.class_names,
                'resize': config.resize,
                'clip_length': config.clip_length,
                'frame_stride': config.frame_stride,
                'train_clips_per_class': config.train_clips_per_class,
                'test_clips_per_class': config.test_clips_per_class,
                'mode': config.mode,
                'input_channels': config.input_channels,
                'batch_size': config.batch_size,
                'epochs': config.epochs,
                'device': config.device,
                'output_dir': str(config.output_dir),
            }
        }
    )
)

## 2. Notebook Helpers

The next cell contains the pipeline logic that would otherwise be scattered across multiple modules: cache-aware frame extraction, optical flow computation, official KTH person splits, and dataset classes.

Two implementation details matter for reruns:

- If frames already exist for a selected video, extraction is skipped.
- If optical-flow `.npy` files already exist for a selected video, flow computation is skipped.

In [ ]:
PERSON_PATTERN = re.compile(r'person(?P<id>\d{2})_')
TRAIN_PERSONS = set(range(1, 17))
TEST_PERSONS = set(range(17, 26))
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32).view(3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32).view(3, 1, 1)
FLOW_SCALE = 20.0


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def extract_person_id(name: str) -> int:
    match = PERSON_PATTERN.search(name)
    if match is None:
        raise ValueError(f'Could not parse KTH person id from {name!r}')
    return int(match.group('id'))


def ensure_kth_downloaded(config: ActionFlowConfig) -> Path:
    script_path = config.download_script.resolve()
    if not script_path.exists():
        raise FileNotFoundError(f'Download script not found: {script_path}')
    subprocess.run(['bash', str(script_path)], check=True)
    raw_root = config.raw_root
    total_videos = sum(len(list((raw_root / class_name).glob('*.avi'))) for class_name in config.class_names)
    if total_videos <= 0:
        raise FileNotFoundError(f'No AVI files found under {raw_root}')
    return raw_root


def summarize_raw_videos(raw_root: Path, class_names: tuple[str, ...]) -> pd.DataFrame:
    rows = []
    for class_name in class_names:
        rows.append({'class': class_name, 'avi_files': len(list((raw_root / class_name).glob('*.avi')))})
    return pd.DataFrame(rows)


def select_raw_videos(
    raw_root: Path,
    class_names: tuple[str, ...],
    train_per_class: int | None,
    test_per_class: int | None,
) -> list[Path]:
    selected: list[Path] = []
    for class_name in class_names:
        train_count = 0
        test_count = 0
        for avi_path in sorted((raw_root / class_name).glob('*.avi')):
            person_id = extract_person_id(avi_path.name)
            if person_id in TRAIN_PERSONS and (train_per_class is None or train_count < train_per_class):
                selected.append(avi_path)
                train_count += 1
            elif person_id in TEST_PERSONS and (test_per_class is None or test_count < test_per_class):
                selected.append(avi_path)
                test_count += 1
    return selected


def extract_video_frames(avi_path: Path, output_dir: Path, resize: tuple[int, int]) -> int:
    output_dir.mkdir(parents=True, exist_ok=True)
    capture = cv2.VideoCapture(str(avi_path))
    if not capture.isOpened():
        raise FileNotFoundError(f'Could not open video: {avi_path}')

    total_frames = max(int(capture.get(cv2.CAP_PROP_FRAME_COUNT)), 0)
    existing_frames = sorted(output_dir.glob('frame_*.png'))
    if total_frames > 0 and len(existing_frames) >= total_frames:
        capture.release()
        return len(existing_frames)

    start_index = len(existing_frames)
    if start_index > 0:
        capture.set(cv2.CAP_PROP_POS_FRAMES, start_index)

    height, width = resize
    frame_index = start_index
    while True:
        success, frame = capture.read()
        if not success:
            break
        resized = cv2.resize(frame, (width, height), interpolation=cv2.INTER_AREA)
        cv2.imwrite(str(output_dir / f'frame_{frame_index:05d}.png'), resized)
        frame_index += 1

    capture.release()
    return frame_index


def extract_frames_for_videos(video_paths: list[Path], raw_root: Path, frames_root: Path, resize: tuple[int, int]) -> list[Path]:
    frame_dirs: list[Path] = []
    for video_path in tqdm(video_paths, desc='Frame extraction', unit='video'):
        frame_dir = frames_root / video_path.relative_to(raw_root).with_suffix('')
        extract_video_frames(video_path, frame_dir, resize)
        frame_dirs.append(frame_dir)
    return frame_dirs


def compute_flow(frame_a: np.ndarray, frame_b: np.ndarray) -> np.ndarray:
    gray_a = frame_a if frame_a.ndim == 2 else cv2.cvtColor(frame_a, cv2.COLOR_BGR2GRAY)
    gray_b = frame_b if frame_b.ndim == 2 else cv2.cvtColor(frame_b, cv2.COLOR_BGR2GRAY)
    flow = cv2.calcOpticalFlowFarneback(
        gray_a,
        gray_b,
        None,
        pyr_scale=0.5,
        levels=3,
        winsize=15,
        iterations=3,
        poly_n=5,
        poly_sigma=1.2,
        flags=0,
    )
    return flow.astype(np.float32, copy=False)


def visualize_flow(flow_array: np.ndarray) -> np.ndarray:
    magnitude, angle = cv2.cartToPolar(flow_array[..., 0], flow_array[..., 1], angleInDegrees=False)
    hsv = np.zeros((*flow_array.shape[:2], 3), dtype=np.uint8)
    hsv[..., 0] = np.mod(angle * 90 / np.pi, 180).astype(np.uint8)
    hsv[..., 1] = 255
    hsv[..., 2] = cv2.normalize(magnitude, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    return cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)


def compute_video_flow(frames_dir: Path, flow_dir: Path) -> int:
    frame_paths = sorted(frames_dir.glob('frame_*.png'))
    if len(frame_paths) < 2:
        return 0

    flow_dir.mkdir(parents=True, exist_ok=True)
    expected = len(frame_paths) - 1
    existing = sorted(flow_dir.glob('flow_*.npy'))
    if len(existing) >= expected:
        return expected

    for index, (first, second) in enumerate(zip(frame_paths[:-1], frame_paths[1:], strict=True)):
        output_path = flow_dir / f'flow_{index:05d}.npy'
        if output_path.exists():
            continue
        frame_a = cv2.imread(str(first), cv2.IMREAD_COLOR)
        frame_b = cv2.imread(str(second), cv2.IMREAD_COLOR)
        if frame_a is None or frame_b is None:
            raise FileNotFoundError(f'Could not read frame pair: {first} / {second}')
        np.save(output_path, compute_flow(frame_a, frame_b))
    return expected


def compute_flow_for_videos(frame_dirs: list[Path], frames_root: Path, flow_root: Path) -> list[Path]:
    flow_dirs: list[Path] = []
    for frame_dir in tqdm(frame_dirs, desc='Optical flow', unit='video'):
        flow_dir = flow_root / frame_dir.relative_to(frames_root)
        compute_video_flow(frame_dir, flow_dir)
        flow_dirs.append(flow_dir)
    return flow_dirs


def limit_prepared_dirs(
    directories: list[Path],
    labels: list[int],
    limit_per_class: int | None,
) -> tuple[list[Path], list[int]]:
    if limit_per_class is None:
        return directories, labels

    counts: dict[int, int] = {}
    limited_dirs: list[Path] = []
    limited_labels: list[int] = []
    for directory, label in zip(directories, labels, strict=True):
        count = counts.get(label, 0)
        if count >= limit_per_class:
            continue
        counts[label] = count + 1
        limited_dirs.append(directory)
        limited_labels.append(label)
    return limited_dirs, limited_labels


def get_train_test_split(
    prepared_root: Path,
    class_names: tuple[str, ...],
    train_per_class: int | None,
    test_per_class: int | None,
) -> tuple[list[Path], list[int], list[Path], list[int]]:
    train_dirs: list[Path] = []
    train_labels: list[int] = []
    test_dirs: list[Path] = []
    test_labels: list[int] = []
    for label, class_name in enumerate(class_names):
        class_root = prepared_root / class_name
        if not class_root.exists():
            continue
        class_dirs = sorted(path for path in class_root.iterdir() if path.is_dir())
        class_train_dirs = [path for path in class_dirs if extract_person_id(path.name) in TRAIN_PERSONS]
        class_test_dirs = [path for path in class_dirs if extract_person_id(path.name) in TEST_PERSONS]
        if train_per_class is not None:
            class_train_dirs = class_train_dirs[:train_per_class]
        if test_per_class is not None:
            class_test_dirs = class_test_dirs[:test_per_class]
        train_dirs.extend(class_train_dirs)
        train_labels.extend([label] * len(class_train_dirs))
        test_dirs.extend(class_test_dirs)
        test_labels.extend([label] * len(class_test_dirs))
    return train_dirs, train_labels, test_dirs, test_labels


def select_clip_indices(length: int, clip_length: int, frame_stride: int, train: bool) -> list[int]:
    required_span = 1 + (clip_length - 1) * frame_stride
    if length >= required_span:
        max_start = length - required_span
        start = random.randint(0, max_start) if train else max_start // 2
        return [start + offset * frame_stride for offset in range(clip_length)]
    return [min(offset * frame_stride, length - 1) for offset in range(clip_length)]


class FlowClipDataset(Dataset[tuple[torch.Tensor, int]]):
    def __init__(self, video_dirs: list[Path], labels: list[int], config: ActionFlowConfig, train: bool) -> None:
        self.video_dirs = video_dirs
        self.labels = labels
        self.config = config
        self.train = train

    def __len__(self) -> int:
        return len(self.video_dirs)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, int]:
        flow_paths = sorted(self.video_dirs[index].glob('flow_*.npy'))
        clip_indices = select_clip_indices(len(flow_paths), self.config.clip_length, self.config.frame_stride, self.train)
        flows = [np.load(flow_paths[position]).astype(np.float32, copy=False) for position in clip_indices]
        flow_clip = np.stack(flows, axis=0)
        if self.train and random.random() < 0.5:
            flow_clip = np.flip(flow_clip, axis=2).copy()
            flow_clip[..., 0] *= -1.0
        flow_clip = np.clip(flow_clip / FLOW_SCALE, -1.0, 1.0)
        tensor = torch.from_numpy(flow_clip.transpose(0, 3, 1, 2).reshape(self.config.input_channels, *self.config.resize))
        return tensor.float(), self.labels[index]


class RGBClipDataset(Dataset[tuple[torch.Tensor, int]]):
    def __init__(self, video_dirs: list[Path], labels: list[int], config: ActionFlowConfig, train: bool) -> None:
        self.video_dirs = video_dirs
        self.labels = labels
        self.config = config
        self.train = train

    def __len__(self) -> int:
        return len(self.video_dirs)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, int]:
        frame_paths = sorted(self.video_dirs[index].glob('frame_*.png'))
        clip_indices = select_clip_indices(len(frame_paths), self.config.clip_length, self.config.frame_stride, self.train)
        center_index = clip_indices[len(clip_indices) // 2]
        frame = cv2.imread(str(frame_paths[center_index]), cv2.IMREAD_COLOR)
        if frame is None:
            raise FileNotFoundError(frame_paths[center_index])
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        if self.train and random.random() < 0.5:
            frame = np.flip(frame, axis=1).copy()
        tensor = torch.from_numpy(frame.transpose(2, 0, 1)).float() / 255.0
        tensor = (tensor - IMAGENET_MEAN) / IMAGENET_STD
        return tensor, self.labels[index]


def evaluate_model_inline(
    model: torch.nn.Module,
    data_loader: DataLoader,
    class_names: tuple[str, ...],
    device: str,
    output_dir: Path,
    mode: str,
) -> dict[str, object]:
    model.eval()
    all_preds: list[int] = []
    all_labels: list[int] = []
    with torch.no_grad():
        for inputs, labels in data_loader:
            logits = model(inputs.to(device))
            all_preds.extend(logits.argmax(dim=1).cpu().tolist())
            all_labels.extend(labels.tolist())

    accuracy = compute_accuracy(all_preds, all_labels)
    confusion = compute_confusion_matrix(all_preds, all_labels, class_names)
    report = classification_report(all_preds, all_labels, class_names)
    figure = plot_confusion_matrix(confusion, class_names, output_dir / f'confusion_matrix_{mode}.png')
    display(figure)
    plt.close(figure)

    metrics_payload = {
        'accuracy': accuracy,
        'confusion_matrix': confusion.tolist(),
        'classification_report': report,
    }
    with (output_dir / f'metrics_{mode}.json').open('w', encoding='utf-8') as handle:
        json.dump(metrics_payload, handle, indent=2)

    print(f'Validation accuracy: {accuracy:.4f}')
    display(pd.DataFrame(report).transpose().round(3))
    return metrics_payload

## 3. Download and Inspect the Raw Dataset

This notebook assumes `download_kth.sh` is the source of truth for dataset setup. The script downloads any missing class archives into `data/kth/raw` and skips classes that already have AVI files, so rerunning the notebook does not redownload completed data.

After the script runs, the summary below confirms what is available before preprocessing starts.

In [ ]:
seed_everything(config.seed)
raw_root = ensure_kth_downloaded(config)
frames_root = config.data_root / 'frames'
flow_root = config.data_root / 'flow'

print(f'Using raw dataset at: {raw_root}')
display(summarize_raw_videos(raw_root, config.class_names))

selected_raw_videos = select_raw_videos(
    raw_root,
    config.class_names,
    train_per_class=config.train_clips_per_class,
    test_per_class=config.test_clips_per_class,
)
print(f'Selected {len(selected_raw_videos)} videos for this notebook run.')

## 4. Frame Extraction

Each selected `.avi` is decoded into resized `.png` frames under `data/kth/frames/{class}/{video_name}`. The extraction function is resumable, so rerunning the notebook simply skips videos that already have the full frame cache.

After extraction, the notebook renders a few example frames inline.

In [ ]:
selected_frame_dirs = extract_frames_for_videos(selected_raw_videos, raw_root, frames_root, config.resize)
print(f'Prepared {len(selected_frame_dirs)} frame directories under {frames_root}.')

example_frame_dir = selected_frame_dirs[0]
example_frame_paths = sorted(example_frame_dir.glob('frame_*.png'))[:3]
figure, axes = plt.subplots(1, len(example_frame_paths), figsize=(15, 4))
for axis, frame_path in zip(axes, example_frame_paths, strict=True):
    frame = cv2.cvtColor(cv2.imread(str(frame_path)), cv2.COLOR_BGR2RGB)
    axis.imshow(frame)
    axis.set_title(frame_path.name)
    axis.axis('off')
figure.suptitle(f'Sample extracted frames: {example_frame_dir.name}')
display(figure)
plt.close(figure)

## 5. Dense Optical Flow

For each pair of consecutive frames, the notebook computes dense Farneback optical flow and caches it as `flow_XXXXX.npy` with shape `(H, W, 2)`. Just like extraction, this step is resumable and only fills in missing flow files.

The visualization below shows what the HSV optical-flow rendering looks like for one cached frame pair.

In [ ]:
selected_flow_dirs = compute_flow_for_videos(selected_frame_dirs, frames_root, flow_root)
print(f'Prepared {len(selected_flow_dirs)} flow directories under {flow_root}.')

example_flow_dir = selected_flow_dirs[0]
example_flow_paths = sorted(example_flow_dir.glob('flow_*.npy'))
example_flow = np.load(example_flow_paths[0])
frame_a = cv2.cvtColor(cv2.imread(str(example_frame_paths[0])), cv2.COLOR_BGR2RGB)
frame_b = cv2.cvtColor(cv2.imread(str(example_frame_paths[1])), cv2.COLOR_BGR2RGB)
flow_hsv = visualize_flow(example_flow)

figure, axes = plt.subplots(1, 3, figsize=(16, 4))
for axis, image, title in zip(
    axes,
    [frame_a, frame_b, flow_hsv],
    ['Frame t', 'Frame t+1', 'Optical flow HSV'],
    strict=True,
):
    axis.imshow(image)
    axis.set_title(title)
    axis.axis('off')
figure.suptitle(f'Optical flow example: {example_flow_dir.name}')
display(figure)
plt.close(figure)

## 6. Dataset Loading

The official KTH split is person-based: persons `01-16` for training and `17-25` for testing. In `flow` mode, each item is a stack of `clip_length` consecutive optical-flow fields shaped into `(2T, H, W)`. In `rgb` mode, each item is the center frame from a clip.

The notebook keeps the selected subset balanced per class so the run stays practical on CPU while still exercising the full real-data pipeline.

In [ ]:
prepared_root = flow_root if config.mode == 'flow' else frames_root
train_dirs, train_labels, test_dirs, test_labels = get_train_test_split(
    prepared_root,
    config.class_names,
    train_per_class=config.train_clips_per_class,
    test_per_class=config.test_clips_per_class,
)

dataset_cls = FlowClipDataset if config.mode == 'flow' else RGBClipDataset
train_dataset = dataset_cls(train_dirs, train_labels, config=config, train=True)
test_dataset = dataset_cls(test_dirs, test_labels, config=config, train=False)

train_loader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    num_workers=config.num_workers,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    num_workers=config.num_workers,
)

sample_batch, sample_labels = next(iter(train_loader))
print(f'Train videos: {len(train_dataset)} | Test videos: {len(test_dataset)}')
print(f'Batch tensor shape: {tuple(sample_batch.shape)}')
print(f'Batch labels: {sample_labels.tolist()}')

## 7. Model Building and Training

The notebook imports the ResNet18 builder from the project module. In flow mode, the first convolution is automatically inflated from RGB weights to `2 * clip_length` channels using the same weight-replication rule documented in the source file.

Training uses the shared `Trainer` class so the notebook stays readable while still saving the best checkpoint to disk.

In [ ]:
model = build_resnet18_flow(
    num_classes=config.num_classes,
    input_channels=config.input_channels,
    pretrained=config.pretrained_backbone,
)
trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=test_loader,
    config=config,
    device=config.device,
)
history = trainer.train()

training_figure = plot_training_curves(
    history,
    save_path=config.output_dir / f'training_curves_{config.mode}.png',
)
display(training_figure)
plt.close(training_figure)

checkpoint_path = config.output_dir / f'best_{config.mode}.pt'
print(f'Best checkpoint: {checkpoint_path}')

## 8. Evaluation and Inline Plots

For evaluation, the notebook reloads the best checkpoint, runs inference on the held-out test split, computes accuracy plus a full classification report, and renders the confusion matrix inline. The same artifacts are also saved to `outputs/notebook` for later comparison.

In [ ]:
checkpoint = torch.load(checkpoint_path, map_location=config.device)
best_model = build_resnet18_flow(
    num_classes=config.num_classes,
    input_channels=config.input_channels,
    pretrained=False,
)
best_model.load_state_dict(checkpoint['model_state_dict'])
best_model.to(config.device)

metrics_payload = evaluate_model_inline(
    model=best_model,
    data_loader=test_loader,
    class_names=config.class_names,
    device=config.device,
    output_dir=config.output_dir,
    mode=config.mode,
)

metrics_payload

## 9. What to Change Next

This notebook is meant to be rerun top-to-bottom. The expensive preprocessing stages are cache-aware, so repeated runs should spend time on training and evaluation rather than recomputing frames or optical flow.

Useful next knobs:

- Set `config.mode = 'rgb'` to run the RGB baseline.
- Increase `train_clips_per_class` and `test_clips_per_class` for a larger real-data run.
- Increase `epochs` for a more meaningful experiment once the pipeline is behaving the way you want.